### First Streaming Query


databricks => https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/1885200041052250?o=7474657872577658#command/5062569379577639


aula => https://customer-academy.databricks.com/learn/courses/3959/stream-processing-and-analysis-with-apache-spark/lessons/36564/demo-window-aggregation-in-spark-structured-streaming?hash=3e063f193bec2e8ca7cdf56ffbdced1657bcc087&generated_by=409740



https://community.databricks.com/t5/learning-events/virtual-learning-festival-10-october-31-october-2025/ev-p/127652


https://community.databricks.com/t5/learning-events/advanced-learning-festival-15-june-06-july-2026/ev-p/156932

# 1.2 Demo - Your First Streaming Query

**Duration:** ~10 minutes

This demo builds a complete streaming query in a handful of cells using the built-in `rate` source, which generates rows at a fixed rate with no external dependencies. It is the smallest possible end-to-end example of the pattern you will use throughout the course: read a stream, transform it, write it to a sink, and stop it cleanly.

## Learning Objectives

By the end of this demo, you will be able to:

* Create a streaming DataFrame with `spark.readStream` and confirm it is streaming

> **REQUIRED - SELECT CLASSIC COMPUTE**
> 
> Before starting this notebook, select your Classic Compute cluster from the compute drop-down at the top-right of the notebook. Be aware that Serverless is selected by default.
> 
> **Note:** This course was developed and tested on Databricks Runtime 17.3 LTS (Apache Spark 4.0). The demos and labs in this course run continuous and triggered streaming queries and reference the Spark UI, which require Classic Compute.

## A. Create a Streaming DataFrame

The rate source generates rows containing a timestamp and an incrementing value at a configurable rows-per-second rate. It exists purely for testing and learning - no setup, no credentials, no data files.

Processamento Batch (Lote): É como ler um livro que já está escrito do princípio ao fim. Tens um conjunto de dados fixo (um ficheiro CSV, uma tabela estática), o Spark processa tudo de uma vez e o trabalho termina.

Streaming (Fluxo Contínuo): É como olhar para uma torneira aberta ou para um canal de notícias em direto. Os dados nunca param de chegar. O Spark fica a escutar ativamente e, assim que novos dados aparecem, processa-os imediatamente em tempo real.

In [0]:
# Create a Streaming DataFrame from the Rate Source
rate_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

print(f"IsStreaming: {rate_df.isStreaming}")
rate_df.printSchema()

IsStreaming: True
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)



O que faz: Cria uma fonte de dados "falsa" gerada pelo próprio Spark, estritamente para testes. Ela cospe automaticamente 5 linhas por segundo, criando um contador a subir (0, 1, 2, 3, 4...) acompanhado por uma coluna de data/hora (timestamp). Como os dados nunca acabam, o fluxo é infinito.

In [0]:
# display() on this compute cannot restart a stream from an existing checkpoint,
# so give each run its own checkpoint subdirectory.
import time

display(rate_df, checkpointLocation=f"/Volumes/workspace/intro_spark/output/checkpoints/rate_demo/{int(time.time())}")

Checkpointing to /Volumes/workspace/intro_spark/output/checkpoints/rate_demo/1794704440


Note the `isStreaming` property: the DataFrame looks and behaves like any other DataFrame, but it is bound to an unbounded source. Nothing is running yet - like all Spark transformations, this is lazy.

## B. Transform the Stream

Streaming DataFrames support the same transformations as static ones. Here we derive a column the same way we would in a batch job.

In [0]:
# Apply a Transformation to the Stream
from pyspark.sql.functions import col

transformed_df = rate_df.withColumn(
    "double_value", col("value") * 2
)

In [0]:
display(transformed_df, checkpointLocation=f"/Volumes/workspace/intro_spark/output/checkpoints/rate_demo_transformed{int(time.time())}")

Checkpointing to /Volumes/workspace/intro_spark/output/checkpoints/rate_demo_transformed1794704481


A Transformação em Tempo Real:

Assim que cada número sai da torneira do rate, o Spark aplica instantaneamente a regra de negócio: verifica se o número é par (even) ou ímpar (odd), criando a coluna parity em tempo real para cada linha nova.

In [0]:
# Apply a Transformation to the Stream
from pyspark.sql.functions import col, when

labeled_df = rate_df.withColumn(
    "parity",
    when((col("value") % 2) == 0, "even").otherwise("odd")
)

## C. Run the Streaming Query

Calling display() on a streaming DataFrame starts a streaming query and renders its results live in the notebook. Let it run for ten seconds or so and watch new rows appear, then click the stop icon on the cell (or run the final cell) to stop the query.

In [0]:
# Display the Stream
# display() on this compute cannot restart a stream from an existing checkpoint,
# so give each run its own checkpoint subdirectory.
import time

display(labeled_df, checkpointLocation=f"/Volumes/workspace/intro_spark/output/checkpoints/rate_demo_labeled/{int(time.time())}")

Checkpointing to /Volumes/workspace/intro_spark/output/checkpoints/rate_demo_labeled/1794704493




Abre uma tabela interativa no ecrã do Databricks que se atualiza sozinha segundo a segundo. Vês as linhas a aparecer dinamicamente na tabela sem precisares de voltar a executar a célula.

In [0]:
# Display the Stream with an explicit checkpoint location
#display(labeled_df, checkpointLocation="/tmp/checkpoint_stream")

Checkpointing to /tmp/checkpoint_stream


## D. Stop All Active Streams 


  Streaming queries run until stopped - they do not end on their own. Always stop queries when you are done with them, or they will hold cluster resources indefinitely. 

In [0]:
# 1. Parar quaisquer streams anteriores que possam estar a bloquear
#for s in spark.streams.active:
#    s.stop()

# 2. Executar o display com o checkpoint explícito usando dbfs:/
#display(labeled_df, checkpointLocation="dbfs:/tmp/stream_checkpoint_labeled")

In [0]:
# Stop All Active Streaming Queries
for query in spark.streams.active:
    query.stop()

print(f"Active streaming queries: {len(spark.streams.active)}")


Active streaming queries: 0



## Summary

* `spark.readStream` creates a streaming DataFrame; the `rate` source needs no setup and is ideal for experiments
* Streaming DataFrames are transformed with the same API as static DataFrames, and remain lazy until a query starts
* `display()` starts an implicit streaming query with live results in the notebook
* Streaming queries run until explicitly stopped — `spark.streams.active` lists them and `query.stop()` ends them

In the next section you will apply this same pattern to a realistic file-based stream with explicit sinks, triggers, and output modes.

In [0]:
#Muda para Classic Compute: Clica no menu pendente onde diz Serverless no canto superior direito do notebook e seleciona o teu cluster clássico (por exemplo, o spark-cluster).

# para visualizar os displays

In [0]:
# página 10

In [0]:
# No Serverless o display() de um DataFrame de streaming só faz um snapshot único
# (AvailableNow) que apanha ~0 linhas da fonte rate — a tabela interativa nunca aparece.
# Para VER a tabela (timestamp, value, parity) simulamos a fonte rate em modo batch:
# 20 linhas a 5 linhas/segundo (0.2s entre linhas), com a mesma transformação da Célula 12.

import time
from pyspark.sql import functions as F

simulated_rate_df = (
    spark.range(0, 20)
    # 5 linhas/segundo => 0.2s entre linhas, tal como a fonte rate
    .withColumn("timestamp", (F.lit(time.time()) + F.col("id") * 0.2).cast("timestamp"))
    .withColumn("value", F.col("id"))
    .withColumn("parity", F.when(F.col("value") % 2 == 0, "even").otherwise("odd"))
    .select("timestamp", "value", "parity")
)

display(simulated_rate_df)

timestamp,value,parity
2026-11-15T01:17:07.163Z,0,even
2026-11-15T01:17:07.363Z,1,odd
2026-11-15T01:17:07.563Z,2,even
2026-11-15T01:17:07.763Z,3,odd
2026-11-15T01:17:07.963Z,4,even
2026-11-15T01:17:08.163Z,5,odd
2026-11-15T01:17:08.363Z,6,even
2026-11-15T01:17:08.563Z,7,odd
2026-11-15T01:17:08.763Z,8,even
2026-11-15T01:17:08.963Z,9,odd
